# MUSANGA V3.1 · NB05 — Satellite predictor exploration: Planet NICFI monthly mosaics

**Purpose:** Cache *individual monthly* Planet NICFI / Tropical Forest Observatory 4-band mosaics at the original ~4.77 m nominal pixel resolution for exploratory satellite predictors (texture, within-cell heterogeneity, canopy spatial context). **No temporal averaging, medians, or composite generation.**

**Source of truth:** `Final/Outputs/V3_1/00_input_audit/00_input_manifest.json` (NB00) + `ROI.shp`, with site/mosaic associations inherited from the V3.1 work. Four quarterly representatives: **March (Q1), June (Q2), September (Q3), December (Q4)** of each forest's UAV reference year. These are *month-level reference periods*, not necessarily the exact UAV flight dates. `Yoko = 2020` has no monthly NICFI imagery in March or June; these are **reported as missing and never silently substituted** with a biannual product or another year.

**Study AOI:** `ROI.shp` ∩ each original RGB orthomosaic **bounding footprint**, unioned by forest, then buffered **250 m in metric CRS**. MOS uses MOS1–2 and MOS5–6; MOS3–4 is excluded from the reference-model registry. The source footprint here means the mosaic rectangular bounding extent; the AOI is principally defined by the ROI polygons, not the entire RGB raster extent.

**Cache:** a dedicated **nested Google Drive folder** (`Final/Outputs/V3_1/05_sat_predictor_exploration/input_layers/Musanga_V3_1_05_Planet_NICFI_Monthly`) for exported, potentially multipart GeoTIFFs. Versioned metadata / separate nested-cache export-task registry lives under `Final/Outputs/V3_1/05_sat_predictor_exploration/`. Each export uses a fixed, identifiable image ID, unchanged DN bands (`B,G,R,N` scaled 0–10000), UTM 35N and `fileDimensions=4096` to split large exports. Export is not an acquisition-date selection of source PlanetScope scenes: each monthly product is already a Planet mosaic assembled from scenes.

**Run:** settings → authenticate → inspect geometry → monthly catalogue audit → submit a few exports → monitor → verify file cache. This notebook never trains a model, modifies the V3/V3.1 source inputs, or claims completed downloads until Earth Engine reports completion and GeoTIFF files are present.

## 00 · Settings: explicit site-year and quarter-month decisions

In [ ]:
# Edit only these settings for normal use.
EE_PROJECT = 'musanga-511110'  # Same Earth Engine Cloud project as V3.1 NB04/04A.
YEAR_BY_SITE = {'Baego': 2023, 'MOS': 2024, 'Yangambi': 2024, 'Yoko': 2020}
QUARTER_MONTHS = {'Q1': 3, 'Q2': 6, 'Q3': 9, 'Q4': 12}
BUFFER_M = 250.0
EXPORT_CRS = 'EPSG:32635'  # Forest sites around Kisangani; metric output raster.
EXPORT_SCALE_M = 4.77
FILE_DIMENSIONS = 4096    # Creates multiple GeoTIFF pieces for large AOIs.
DRIVE_EXPORT_FOLDER = 'Musanga_V3_1_05_Planet_NICFI_Monthly'  # Unique folder name; precreated inside Final/Outputs/V3_1/05_sat_predictor_exploration/input_layers/.
RUN_SITES = ['Baego', 'MOS', 'Yangambi', 'Yoko']
SUBMIT_EXPORTS = False    # Audit selection first, then set True and rerun submission cell.
MAX_NEW_TASKS_PER_RUN = 4 # Avoid queuing a huge set of full-resolution exports at once.
SKIP_EXISTING_OR_ACTIVE = True
SOURCE_COLLECTION = 'projects/planet-nicfi/assets/basemaps/africa'
BANDS = ['B', 'G', 'R', 'N']
print('Quarter months:', QUARTER_MONTHS, ' | submitting:', SUBMIT_EXPORTS)

## 01 · Colab environment, Drive and Earth Engine

**Important:** This notebook does not run `pip` or change `google-auth`. If you previously ran the old cell, choose **Runtime → Disconnect and delete runtime**, reconnect, and then **Runtime → Run all**. You will be asked to authorize Earth Engine and, unless entered in Settings, provide your Google Cloud **project ID**.

In [ ]:
# Colab includes these packages. Do NOT pip-install or upgrade google-auth / earthengine-api here.
# If packages were upgraded earlier, use Runtime > Disconnect and delete runtime, then reconnect.
from google.colab import drive
import ee, geopandas as gpd, rasterio, pandas as pd
import json, hashlib, os, calendar, datetime, re
from pathlib import Path
from shapely.geometry import box, mapping
from shapely.ops import unary_union
from IPython.display import display

drive.mount('/content/drive')
ROOT = next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                             Path('/content/drive/MyDrive/Musanga/Final')]
             if p.is_dir()), None)
assert ROOT is not None, 'Musanga/Final not found in Google Drive.'
BASE = ROOT/'Outputs/V3_1'
MANIFEST_PATH = BASE/'00_input_audit/00_input_manifest.json'
assert MANIFEST_PATH.is_file(), 'Run NB00 audit, or correct MANIFEST_PATH.'
CACHE_META = BASE/'05_sat_predictor_exploration'
CACHE_META.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = CACHE_META/'input_layers'
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DRIVE = INPUT_ROOT/DRIVE_EXPORT_FOLDER
CACHE_DRIVE.mkdir(parents=True, exist_ok=True)
# Earth Engine toDrive(folder=...) resolves by FOLDER NAME, not by full path.
# The unique precreated folder under the Musanga workspace avoids ambiguous names.
# Google Drive folder indexing may be delayed: check actual target on first export.
# Explicit user authentication, followed by initialization using a Cloud project ID.
# Earth Engine now requires the project even when login succeeds.
if not EE_PROJECT or EE_PROJECT == 'your-ee-project-id':
    EE_PROJECT = input('Enter your Earth Engine Google Cloud PROJECT ID: ').strip()
if not EE_PROJECT:
    raise ValueError('Project ID is required. Find it in the Earth Engine Code Editor project selector.')
print('Authenticating Earth Engine for project:', EE_PROJECT)
ee.Authenticate(auth_mode='colab')
ee.Initialize(project=EE_PROJECT)
COLLECTION = ee.ImageCollection(SOURCE_COLLECTION)
print('ROOT:', ROOT)
print('Source manifest:', MANIFEST_PATH)
print('Cache metadata:', CACHE_META)
print('Planet raw input layers:', CACHE_DRIVE)
print('Planet collection size (permission test):', COLLECTION.size().getInfo())

## 02 · Site-specific ROI × orthomosaic footprints + 250 m buffer

In [ ]:
A = json.loads(MANIFEST_PATH.read_text())
ROI_PATH = Path(A['roi'])
assert ROI_PATH.exists(), f'Missing authoritative ROI: {ROI_PATH}'
ROI = gpd.read_file(ROI_PATH)
assert ROI.crs is not None and len(ROI) > 0
ROI = ROI[ROI.geometry.notna() & ~ROI.geometry.is_empty].copy()
# Scope to the same 5 source RGB mosaics used by V3.1 NB02B.
MOSAIC_ALLOW = {
 'Baego.tif': 'Baego',
 'Mos_1_2_transparent_mosaic_group1_georef.tif': 'MOS',
 'Mos_5_6_transparent_mosaic_group1_georef.tif': 'MOS',
 'Yangambi_georef.tif': 'Yangambi',
 'Yoko_kvo_georef.tif': 'Yoko',
}
records = []
intersections = {site:[] for site in YEAR_BY_SITE}
for row in A['mosaics']:
    name = row['file']
    if name not in MOSAIC_ALLOW:
        continue
    path = Path(row['path'])
    assert path.exists(), f'Missing RGB mosaic: {path}'
    site = MOSAIC_ALLOW[name]
    with rasterio.open(path) as ds:
        assert ds.crs, f'No raster CRS: {name}'
        # Raster's rectangular geo-referenced *bounding* footprint, not the valid-pixel mask.
        raster_envelope = box(*ds.bounds)
        r = ROI.to_crs(ds.crs)
        roi_union = unary_union([g for g in r.geometry if g is not None and not g.is_empty])
        clipped = roi_union.intersection(raster_envelope)
        records.append({'site':site, 'mosaic':name, 'raster_crs':str(ds.crs),
                        'roi_intersection_m2_in_raster_crs':float(clipped.area),
                        'raster_width':ds.width,'raster_height':ds.height})
        if not clipped.is_empty:
            # Store pieces in a shared geographic CRS until per-site metric buffering.
            gg = gpd.GeoSeries([clipped], crs=ds.crs).to_crs('EPSG:4326').iloc[0]
            intersections[site].append(gg)

display(pd.DataFrame(records))
AOI = {}
audit = []
for site in RUN_SITES:
    assert intersections[site], f'No ROI/footprint intersection for {site}; inspect ROI.shp and input registry.'
    united_wgs84 = unary_union(intersections[site])
    base = gpd.GeoSeries([united_wgs84], crs='EPSG:4326')
    # For these DRC forests projected UTM 35N is appropriate; estimate per-site to avoid assuming.
    metric_crs = base.estimate_utm_crs()
    assert metric_crs is not None
    projected = base.to_crs(metric_crs).iloc[0]
    buffered = projected.buffer(BUFFER_M)
    # Light simplification (≤1 m) controls request size; actual 250 m buffer remains explicit.
    buffered = buffered.simplify(1.0, preserve_topology=True)
    assert buffered.is_valid and not buffered.is_empty
    buffered_wgs = gpd.GeoSeries([buffered], crs=metric_crs).to_crs('EPSG:4326').iloc[0]
    AOI[site] = {'shapely':buffered, 'crs':metric_crs,
                 'ee':ee.Geometry(mapping(buffered_wgs), geodesic=False),
                 'geojson':mapping(buffered_wgs)}
    audit.append({'site':site,'uav_reference_year':YEAR_BY_SITE[site],
                  'source_mosaics':len(intersections[site]),
                  'roi_ha':round(projected.area/1e4,3),
                  'buffered_ha':round(buffered.area/1e4,3),
                  'buffer_m':BUFFER_M,'metric_crs':str(metric_crs),
                  'approx_uint16_fourband_GB_per_month':round(buffered.area/(EXPORT_SCALE_M**2)*8/1e9,2)})

display(pd.DataFrame(audit))
(CACHE_META/'05_site_aoi_audit.csv').write_text(pd.DataFrame(audit).to_csv(index=False))
(CACHE_META/'05_site_aoi_geometries.geojson').write_text(json.dumps({
 'type':'FeatureCollection','features':[{'type':'Feature','properties':{'site':s},'geometry':AOI[s]['geojson']} for s in RUN_SITES]
}))
print('Site-specific ROI + 250 m buffer geometry audited; no imagery exported yet.')

## 03 · Exact one-month selection: do not aggregate or replace missing months

In [ ]:
def month_dates(year, month):
    start = datetime.date(year, month, 1)
    end = datetime.date(year + (month == 12), (month % 12) + 1, 1)
    return start.isoformat(), end.isoformat()

selected = {}
rows = []
for site in RUN_SITES:
    year = YEAR_BY_SITE[site]
    for q, month in QUARTER_MONTHS.items():
        begin, end = month_dates(year, month)
        subset = (COLLECTION.filterDate(begin, end)
                   .filterBounds(AOI[site]['ee'])
                   .filter(ee.Filter.eq('cadence','monthly')))
        number = int(subset.size().getInfo())
        record = {'site':site,'year':year,'quarter':q,'month':month,
                  'requested_month_start':begin, 'requested_month_end_exclusive':end,
                  'matching_monthly_images':number,
                  'status':'available' if number == 1 else ('missing' if number == 0 else 'ambiguous')}
        if number == 1:
            image = ee.Image(subset.first())
            info = ee.Dictionary({
                'id':image.id(),
                'index':image.get('system:index'),
                'time_start':image.get('system:time_start'),
                'time_end':image.get('system:time_end'),
                'cadence':image.get('cadence'),
                'bands':image.bandNames()
            }).getInfo()
            record.update({'image_id':info.get('id'),'image_index':info.get('index'),
                           'time_start_ms':info.get('time_start'), 'time_end_ms':info.get('time_end'),
                           'cadence':info.get('cadence'), 'bands':','.join(info.get('bands',[]))})
            assert set(BANDS).issubset(set(info.get('bands',[])))
            selected[(site,q)] = {'image':image, 'id':info.get('id'), 'year':year, 'month':month}
        rows.append(record)
SELECTION = pd.DataFrame(rows)
display(SELECTION)
SELECTION.to_csv(CACHE_META/'05_planet_monthly_selection.csv',index=False)
assert not (SELECTION.status == 'ambiguous').any(), 'Ambiguous month: inspect catalogue before exporting.'
print('Selected:',len(selected),'of',len(SELECTION),'requested month/site combinations.')
print('Expected Yoko 2020: Q1 and Q2 missing (monthly archive begins September 2020).')
print('No temporal substitution; no mosaics combined.')

## 04 · Export to Drive: georeferenced, lossless source-band input layers

In [ ]:
TASK_REGISTRY = CACHE_META/'05_export_registry_nested_input_layers_v2.json'  # Separate from obsolete root-of-Drive tasks.
if TASK_REGISTRY.exists():
    registry = json.loads(TASK_REGISTRY.read_text())
else:
    registry = {}

def save_registry():
    TASK_REGISTRY.write_text(json.dumps(registry, indent=2, sort_keys=True))

def ee_task_state(task_id):
    if not task_id:
        return 'UNKNOWN'
    try:
        res = ee.data.getTaskStatus(task_id)
        return res[0].get('state','UNKNOWN') if res else 'UNKNOWN'
    except Exception as exc:
        return 'UNKNOWN ('+str(exc)[:80]+')'

# Preserve queued/running/finished task handles even if the Colab runtime restarts.
active_by_description = {}
try:
    for t in ee.batch.Task.list():
        st=t.status()
        if st.get('state') in {'READY','RUNNING','COMPLETED'}:
            active_by_description.setdefault(st.get('description'), []).append(st)
except Exception as exc:
    print('Task-list lookup unavailable:', str(exc)[:200])

new_tasks = 0
for site in RUN_SITES:
    for q, month in QUARTER_MONTHS.items():
        key = f'{site}_{YEAR_BY_SITE[site]}_{q}_M{month:02d}'
        if (site,q) not in selected:
            print('MISSING MONTH (not exported):', key)
            continue
        data = selected[(site,q)]
        image_id = data['id']
        name = f'NB05_Planet_{key}_DN4'
        prefix = name
        task_description = 'MUSANGA_V31_NESTED_' + name
        current = registry.get(key,{})
        if current and current.get('image_id') != image_id:
            raise RuntimeError(f'Image changed for cached key {key}; manual review needed.')
        local_parts = list(CACHE_DRIVE.glob(prefix+'*.tif')) if CACHE_DRIVE.exists() else []
        old_state = ee_task_state(current.get('task_id')) if current else 'NOT_SUBMITTED'
        active = active_by_description.get(task_description, [])
        if SKIP_EXISTING_OR_ACTIVE and (old_state in ('READY','RUNNING','COMPLETED') or active or local_parts):
            print('SKIP existing/active:',key,'old task state:',old_state,
                  'files now visible:',len(local_parts))
            if active and not current.get('task_id'):
                current['task_id'] = active[0].id
                current['description'] = task_description
                current['image_id'] = image_id
                registry[key]=current
            continue
        if not SUBMIT_EXPORTS:
            print('READY FOR EXPORT:',key,'→',name)
            continue
        if new_tasks >= MAX_NEW_TASKS_PER_RUN:
            print('Batch cap reached; rerun this cell when tasks finish.')
            continue
        image = data['image'].select(BANDS).toUint16().clip(AOI[site]['ee'])
        task = ee.batch.Export.image.toDrive(
            image=image,
            description=task_description,
            folder=DRIVE_EXPORT_FOLDER,
            fileNamePrefix=prefix,
            region=AOI[site]['ee'],
            crs=EXPORT_CRS,
            scale=EXPORT_SCALE_M,
            fileDimensions=FILE_DIMENSIONS,
            shardSize=256,
            skipEmptyTiles=True,
            maxPixels=1e13,
            fileFormat='GeoTIFF',
            formatOptions={'cloudOptimized':True}
        )
        task.start()
        registry[key] = {
            'site':site, 'quarter':q,'year':data['year'],'month':month,
            'image_id':image_id,'task_id':task.id,'description':task_description,
            'drive_folder':DRIVE_EXPORT_FOLDER, 'prefix':prefix,
            'band_order':BANDS,'pixel_type':'uint16', 'DN_scale_factor':0.0001,
            'crs':EXPORT_CRS, 'nominal_scale_m':EXPORT_SCALE_M,
            'roi_buffer_m':BUFFER_M, 'file_dimensions':FILE_DIMENSIONS,
            'note':'individual monthly NICFI/TFO image; no temporal aggregation'
        }
        save_registry()
        new_tasks += 1
        print('SUBMITTED:',key,'task',task.id)
save_registry()
print('New jobs submitted:',new_tasks,'| registry:',TASK_REGISTRY)

## 05 · Monitor, verify and audit cache; never mistake submitted for downloaded

In [ ]:
summary = []
for key,item in registry.items():
    task_state = ee_task_state(item.get('task_id'))
    prefix=item['prefix']
    files = sorted(CACHE_DRIVE.glob(prefix+'*.tif')) if CACHE_DRIVE.exists() else []
    nbytes=sum(f.stat().st_size for f in files)
    # A large export may be split across many GeoTIFFs; number of parts is variable.
    visible = task_state == 'COMPLETED' and bool(files)
    # File visibility is NOT full radiometric or spatial integrity; Section 06 checks that.
    summary.append({'site_quarter':key,'EE_task_state':task_state,'n_tif_parts':len(files),
                    'total_GB':round(nbytes/1e9,3),'completed_file_visible':visible,
                    'image_id':item['image_id']})
    if task_state == 'FAILED':
        print('FAILED TASK:',key,item.get('task_id'),'— check Earth Engine task details')
AUDIT = pd.DataFrame(summary)
display(AUDIT)
AUDIT.to_csv(CACHE_META/'05_planet_export_status_nested_v2.csv',index=False)
print('Completed + file-visible quarters (run Section 06 for full QA):',int(AUDIT.completed_file_visible.sum()) if len(AUDIT) else 0)
print('Drive folder:',CACHE_DRIVE)
print('CAUTION: exported NICFI monthly mosaics are already multi-scene products;')
print('these cached data contain NO further temporal aggregation by NB05.')

## 06 · Full-scene integrity QA: spatial coverage and radiometry

Scan every GeoTIFF blockwise to distinguish all-zero imagery from zero-filled areas outside an irregular ROI. Report nonzero coverage, positive four-band coverage and values sampled from genuine data. **Do not interpret a top-left zero-valued window as a failed export.**

In [ ]:
# 06 — COMPLETE, BLOCKWISE, FOUR-BAND RASTER QA (never rely on corner pixels)
# Read every raster in blocks; no full-scene memory allocation.
import numpy as np, pandas as pd, rasterio
from pathlib import Path
from IPython.display import display

paths = sorted(CACHE_DRIVE.glob('NB05_Planet_*.tif'))
assert CACHE_DRIVE.is_dir(), CACHE_DRIVE
if not paths:
    print('No GeoTIFF parts visible in expected nested folder:', CACHE_DRIVE)
else:
    qa_rows = []
    for path in paths:
        with rasterio.open(path) as ds:
            assert ds.count == 4 and all(d == 'uint16' for d in ds.dtypes), (path,ds.count,ds.dtypes)
            assert ds.crs is not None, path
            total = ds.width * ds.height
            valid_all = 0
            any_nonzero = 0
            max_band = np.zeros(4, dtype=np.uint16)
            min_positive_band = [None]*4
            good_samples = []
            for _, window in ds.block_windows(1):
                # GEE exports often store off-ROI pixels as zero with no NoData metadata.
                a = ds.read(window=window, masked=False)
                any_valid = np.any(a > 0, axis=0)
                all_valid = np.all(a > 0, axis=0)
                any_nonzero += int(any_valid.sum())
                valid_all += int(all_valid.sum())
                max_band = np.maximum(max_band, a.reshape(4,-1).max(axis=1))
                for b in range(4):
                    positive = a[b][a[b]>0]
                    if positive.size:
                        lo = int(positive.min())
                        min_positive_band[b] = lo if min_positive_band[b] is None else min(lo,min_positive_band[b])
                if len(good_samples)<4 and all_valid.any():
                    rr,cc = np.argwhere(all_valid)[len(np.argwhere(all_valid))//2]
                    good_samples.append((int(window.row_off+rr), int(window.col_off+cc), a[:,rr,cc].tolist()))
            status = 'OK' if any_nonzero>0 and valid_all>0 else 'EMPTY_OR_ZERO'
            qa_rows.append(dict(file=path.name,status=status,crs=str(ds.crs),
                width=ds.width,height=ds.height,mb=round(path.stat().st_size/1e6,2),
                any_nonzero_pct=round(100*any_nonzero/total,2),
                all_four_positive_pct=round(100*valid_all/total,2),
                band_max_B_G_R_N=max_band.tolist(),
                band_min_positive_B_G_R_N=min_positive_band,
                example_valid_pixel=good_samples[0] if good_samples else None))
    qa = pd.DataFrame(qa_rows)
    display(qa)
    qa.to_csv(CACHE_META/'05_planet_full_raster_quality.csv',index=False)
    print('Important: zero is being treated as exported background, not guaranteed NoData.')
    print('If zero-only: check geometry, exported region, EE task details and mask handling.')
    print('If complete EE task has no visible file: check Drive indexing/location, do NOT resubmit blindly.')


## Scientific safeguards and next experiments

1. **Timing:** 2020 Yoko cannot yield four monthly quarters; do not switch to 2021 or biannual archive without declaring a different temporal experiment. Flight date precision is year-only in the V3.1 model registry.
2. **Spatial masking:** Site AOIs are defined as intersection of the global `ROI.shp` polygons with each audited RGB mosaic *bounding box*, merged by forest and buffered 250 m. The raw raster valid-pixel footprint may be smaller than its bounding box. Planet has independent valid-data masks.
3. **Spatial resolution:** Each cached DN band retains four-band Planet information at a 4.77 m **export sample spacing**, not a guarantee of original pixel-grid alignment. Exports use EPSG:32635. Never calculate sub-10 m texture on imagery that was first aggregated to 100 m.
4. **File organisation:** Planet GeoTIFFs are exported into the precreated, uniquely named folder inside `Final/Outputs/V3_1/05_sat_predictor_exploration/input_layers/`. Earth Engine Drive exports resolve folders by name rather than absolute path: verify the first completed file lands here; if Drive indexing lags, stop and relocate any misplaced initial file before continuing.
5. **Radiometry:** The NICFI analytic bands are scaled DN (multiply by 0.0001 for reflectance). There is **no additional harmonization or per-date normalization** here. Inspect monthly radiometry before pooling texture features.
6. **Comparison:** Derive Planet features **inside the spatial CV training folds only**, freeze the same UAV target and folds as NB04, and use strict four-forest site holdout to judge generalization. Validate especially the high-Musanga fraction tail at MOS.
6. **Caching:** GEE Drive exports may create several `*_000...tif` parts; keep them together. The export registry gives task IDs and provenance. A task being READY/RUNNING does not mean files are downloaded.
7. **Future integration:** cached source rasters are for predictor exploration, not yet the final combined satellite feature stack. A next NB05 block can compute GLCM/local variance/edge density at 4.77 m and aggregate summaries to the 100 m target support, with consistent site/fold masks.

**Inputs:** V3.1 NB00 manifest, `ROI.shp`, audited UAV mosaic headers, `projects/planet-nicfi/assets/basemaps/africa`.

**Outputs:** Drive GeoTIFF cache; `05_site_aoi_audit.csv`, `05_site_aoi_geometries.geojson`, `05_planet_monthly_selection.csv`, `05_export_registry.json`, `05_planet_export_status.csv`.